# 🏗️ Low-Level Design (LLD) Mastery - Senior Developer Edition

This notebook focuses on translating requirements into clean classes, contracts, and implementation-ready designs.

## 🎯 What You Will Master

### Core LLD Skills
- **Object modeling**: entities, value objects, aggregates, and boundaries
- **SOLID principles**: extensible and testable architecture
- **Design patterns**: choosing patterns by problem type, not memorization
- **Interface-first design**: dependency inversion and clean contracts
- **Code quality**: readability, maintainability, and operational safety

### Interview Outcomes
- Convert ambiguous problems into clear object models
- Explain why each class and interface exists
- Handle extensibility requirements without major rewrites
- Discuss trade-offs (simplicity vs flexibility, memory vs speed)

### 5-Step LLD Workflow (Use for Every Problem)
1. Clarify scope and expected operations.
2. Identify entities, responsibilities, and invariants.
3. Define interfaces and interaction contracts.
4. Add patterns only where they reduce complexity.
5. Validate with scenarios, edge cases, and tests.

---

**Pro tip:** In interviews, start with a minimal working design, then evolve it for extensibility and scale.

**Real-world LLD examples:** Open **50 Real-World Examples (HLD + LLD)** in the sidebar — Parking Lot, Elevator, LRU Cache, Chess, ATM, and 13 more with class diagrams and flows.

## Chapter 1: SOLID Principles ⭐⭐⭐

**Interview Frequency: Very High** | **Career Impact: Core Senior-Level Signal**

SOLID is not memorization; it is a practical checklist for maintainability and change safety.

### Chapter Objectives
- Recognize violations in real code quickly.
- Refactor designs to improve testability and extensibility.
- Explain trade-offs when strict SOLID adds complexity.

### SOLID Quick Map
- **S** - Single Responsibility Principle
- **O** - Open/Closed Principle
- **L** - Liskov Substitution Principle
- **I** - Interface Segregation Principle
- **D** - Dependency Inversion Principle

### Practice Output
For each principle, write:
1) one broken example,
2) one refactored version,
3) one test scenario proving the improvement.

## 🏗️ Object-Oriented Design - Complete Theory

### **📚 SOLID Principles - Foundation of Good Design**

---

## 1️⃣ **Single Responsibility Principle (SRP)**

**Definition:** A class should have ONE and ONLY ONE reason to change.

```
❌ BAD: God Class (Multiple Responsibilities)

class User:
    def __init__(self, name, email):
        self.name = name
        self.email = email
    
    def save_to_database(self):
        # Database logic - Responsibility 1
        pass
    
    def send_email(self):
        # Email logic - Responsibility 2
        pass
    
    def generate_report(self):
        # Report logic - Responsibility 3
        pass

Problems:
• Change DB? Modify User class
• Change email provider? Modify User class
• Change report format? Modify User class
• Hard to test (mock DB, email, report)
• Hard to reuse (tight coupling)

✅ GOOD: Single Responsibility per Class

class User:
    def __init__(self, name, email):
        self.name = name
        self.email = email

class UserRepository:
    def save(self, user: User):
        # Database logic only
        pass

class EmailService:
    def send_welcome_email(self, user: User):
        # Email logic only
        pass

class ReportGenerator:
    def generate_user_report(self, user: User):
        # Report logic only
        pass

Benefits:
✅ Change DB → Modify only UserRepository
✅ Change email → Modify only EmailService
✅ Easy to test (mock one service at a time)
✅ Reusable (EmailService for all emails)
```

**Interview Example:**
"Design a logger that writes to file and sends critical errors to monitoring service."

❌ Bad:
```python
class Logger:
    def log(self, message, level):
        self.write_to_file(message)
        if level == "CRITICAL":
            self.send_to_monitoring(message)
```

✅ Good:
```python
class Logger:
    def log(self, message, level):
        pass

class FileLogger(Logger):
    def log(self, message, level):
        self.write_to_file(message)

class MonitoringLogger(Logger):
    def log(self, message, level):
        if level == "CRITICAL":
            self.send_to_monitoring(message)

class CompositeLogger(Logger):
    def __init__(self, loggers):
        self.loggers = loggers
    
    def log(self, message, level):
        for logger in self.loggers:
            logger.log(message, level)
```

---

## 2️⃣ **Open/Closed Principle (OCP)**

**Definition:** Software entities should be OPEN for extension but CLOSED for modification.

```
❌ BAD: Modifying existing code for new features

class PaymentProcessor:
    def process(self, payment_type, amount):
        if payment_type == "credit_card":
            # Process credit card
            pass
        elif payment_type == "paypal":
            # Process PayPal
            pass
        elif payment_type == "crypto":  # New feature
            # Process crypto
            pass
        # Every new payment type requires modifying this class!

Problems:
• Violates OCP (modify existing code)
• Risk of breaking existing functionality
• Hard to test (many branches)

✅ GOOD: Extend through inheritance/composition

class PaymentProcessor(ABC):
    @abstractmethod
    def process(self, amount):
        pass

class CreditCardProcessor(PaymentProcessor):
    def process(self, amount):
        # Credit card logic
        pass

class PayPalProcessor(PaymentProcessor):
    def process(self, amount):
        # PayPal logic
        pass

class CryptoProcessor(PaymentProcessor):  # New feature
    def process(self, amount):
        # Crypto logic (NO modification to existing classes!)
        pass

class PaymentService:
    def __init__(self, processor: PaymentProcessor):
        self.processor = processor
    
    def pay(self, amount):
        self.processor.process(amount)

# Usage
service = PaymentService(CreditCardProcessor())
service.pay(100)

service = PaymentService(CryptoProcessor())  # New processor, no code change!
service.pay(100)

Benefits:
✅ Add new payment types without modifying existing code
✅ Existing tests don't need to change
✅ No risk to existing functionality
```

**Real-World Example: Django/Flask Middleware**
```python
# Open for extension (add middleware)
# Closed for modification (framework code untouched)

app.use(AuthenticationMiddleware())
app.use(LoggingMiddleware())
app.use(RateLimitingMiddleware())  # New feature, no framework change
```

---

## 3️⃣ **Liskov Substitution Principle (LSP)**

**Definition:** Objects of a superclass should be replaceable with objects of a subclass without breaking the application.

```
❌ BAD: Subclass violates parent contract

class Bird:
    def fly(self):
        pass

class Sparrow(Bird):
    def fly(self):
        # Can fly
        pass

class Penguin(Bird):
    def fly(self):
        raise Exception("Penguins can't fly!")  # Violates LSP!

# Usage
def make_bird_fly(bird: Bird):
    bird.fly()  # Expect all birds to fly

make_bird_fly(Sparrow())  # Works
make_bird_fly(Penguin())  # Crashes! Violated LSP

Problem: Penguin is a Bird but can't substitute Bird behavior

✅ GOOD: Proper abstraction

class Bird:
    def eat(self):
        pass

class FlyingBird(Bird):
    def fly(self):
        pass

class Sparrow(FlyingBird):
    def fly(self):
        # Can fly
        pass

class Penguin(Bird):  # Not a FlyingBird
    def swim(self):
        # Can swim
        pass

# Usage
def make_bird_fly(bird: FlyingBird):  # Specific contract
    bird.fly()

make_bird_fly(Sparrow())  # Works
make_bird_fly(Penguin())  # Compile error (type safety)

Benefits:
✅ Clear contracts
✅ No surprises at runtime
✅ Type-safe
```

**Interview Example: Rectangle-Square Problem**
```python
❌ BAD:

class Rectangle:
    def set_width(self, width):
        self.width = width
    
    def set_height(self, height):
        self.height = height
    
    def area(self):
        return self.width * self.height

class Square(Rectangle):
    def set_width(self, width):
        self.width = width
        self.height = width  # Maintain square property
    
    def set_height(self, height):
        self.width = height
        self.height = height

# Usage
def test(rectangle: Rectangle):
    rectangle.set_width(5)
    rectangle.set_height(4)
    assert rectangle.area() == 20  # Expect 20

test(Rectangle())  # Works (20)
test(Square())     # Fails! (16, not 20) - Violated LSP

✅ GOOD: Separate abstractions
class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height
    
    def area(self):
        return self.width * self.height

class Square(Shape):
    def __init__(self, side):
        self.side = side
    
    def area(self):
        return self.side * self.side
```

---

## 4️⃣ **Interface Segregation Principle (ISP)**

**Definition:** Clients should not be forced to depend on interfaces they don't use.

```
❌ BAD: Fat Interface

class Worker(ABC):
    @abstractmethod
    def work(self):
        pass
    
    @abstractmethod
    def eat(self):
        pass

class HumanWorker(Worker):
    def work(self):
        # Humans work
        pass
    
    def eat(self):
        # Humans eat
        pass

class RobotWorker(Worker):
    def work(self):
        # Robots work
        pass
    
    def eat(self):
        raise NotImplementedError("Robots don't eat!")  # Forced to implement!

Problem: RobotWorker forced to implement eat() even though it doesn't need it

✅ GOOD: Segregated Interfaces

class Workable(ABC):
    @abstractmethod
    def work(self):
        pass

class Eatable(ABC):
    @abstractmethod
    def eat(self):
        pass

class HumanWorker(Workable, Eatable):  # Implements both
    def work(self):
        pass
    
    def eat(self):
        pass

class RobotWorker(Workable):  # Only implements what it needs
    def work(self):
        pass

# Usage
def manage_work(worker: Workable):
    worker.work()

def manage_break(worker: Eatable):
    worker.eat()

manage_work(HumanWorker())  # Works
manage_work(RobotWorker())  # Works

manage_break(HumanWorker())  # Works
manage_break(RobotWorker())  # Type error (good!)

Benefits:
✅ Classes only implement what they need
✅ No unnecessary dependencies
✅ Easier to maintain
```

**Real-World Example: Database Interfaces**
```python
❌ BAD:
class Database(ABC):
    @abstractmethod
    def connect(self):
        pass
    
    @abstractmethod
    def execute_query(self):
        pass
    
    @abstractmethod
    def execute_transaction(self):
        pass
    
    @abstractmethod
    def execute_stored_procedure(self):
        pass

# NoSQL database forced to implement transactions/stored procedures!

✅ GOOD:
class Connectable(ABC):
    @abstractmethod
    def connect(self):
        pass

class Queryable(ABC):
    @abstractmethod
    def execute_query(self):
        pass

class Transactional(ABC):
    @abstractmethod
    def execute_transaction(self):
        pass

class SQLDatabase(Connectable, Queryable, Transactional):
    pass

class NoSQLDatabase(Connectable, Queryable):  # No transactions
    pass
```

---

## 5️⃣ **Dependency Inversion Principle (DIP)**

**Definition:** 
1. High-level modules should not depend on low-level modules. Both should depend on abstractions.
2. Abstractions should not depend on details. Details should depend on abstractions.

```
❌ BAD: High-level depends on low-level (tight coupling)

class MySQLDatabase:
    def save(self, data):
        # MySQL-specific code
        pass

class UserService:  # High-level
    def __init__(self):
        self.db = MySQLDatabase()  # Depends on concrete implementation!
    
    def create_user(self, user):
        self.db.save(user)

Problems:
• Can't switch to PostgreSQL without changing UserService
• Hard to test (need real MySQL)
• Tight coupling

✅ GOOD: Both depend on abstraction

class Database(ABC):  # Abstraction
    @abstractmethod
    def save(self, data):
        pass

class MySQLDatabase(Database):  # Low-level
    def save(self, data):
        # MySQL-specific code
        pass

class PostgreSQLDatabase(Database):  # Low-level
    def save(self, data):
        # PostgreSQL-specific code
        pass

class UserService:  # High-level
    def __init__(self, db: Database):  # Depends on abstraction!
        self.db = db
    
    def create_user(self, user):
        self.db.save(user)

# Usage
service = UserService(MySQLDatabase())  # Production
service = UserService(PostgreSQLDatabase())  # Easy switch
service = UserService(MockDatabase())  # Testing

Benefits:
✅ Flexible (swap implementations)
✅ Testable (mock the abstraction)
✅ Loose coupling
```

**Dependency Injection (DI)**
```python
# Constructor Injection (preferred)
class OrderService:
    def __init__(self, db: Database, email: EmailService):
        self.db = db
        self.email = email

# Setter Injection
class OrderService:
    def set_database(self, db: Database):
        self.db = db

# Method Injection
class OrderService:
    def create_order(self, order, db: Database):
        db.save(order)
```

**Real-World Example: Framework DI**
```python
# Spring Boot / FastAPI style

@Injectable()
class UserService:
    def __init__(self, db: Database, cache: Cache):
        self.db = db  # Framework injects dependencies
        self.cache = cache

# Configuration
Container.bind(Database, PostgreSQLDatabase)
Container.bind(Cache, RedisCache)
```

---

## 🎨 **Design Patterns Overview**

### **Creational Patterns** (Object creation)

```
1. Singleton - One instance only
   class Database:
       _instance = None
       
       def __new__(cls):
           if cls._instance is None:
               cls._instance = super().__new__(cls)
           return cls._instance

2. Factory - Create objects without specifying exact class
   class ShapeFactory:
       def create(self, type):
           if type == "circle":
               return Circle()
           elif type == "square":
               return Square()

3. Builder - Construct complex objects step by step
   Car().set_engine("V8").set_color("red").build()

4. Prototype - Clone existing objects
   new_object = existing_object.clone()
```

### **Structural Patterns** (Object composition)

```
1. Adapter - Make incompatible interfaces compatible
   class LegacyAdapter:
       def __init__(self, legacy_system):
           self.legacy = legacy_system
       
       def new_method(self):
           return self.legacy.old_method()

2. Decorator - Add behavior without modifying class
   @cache
   @retry
   @log
   def expensive_function():
       pass

3. Proxy - Control access to object
   class ImageProxy:
       def __init__(self, filename):
           self.filename = filename
           self.image = None  # Lazy load
       
       def display(self):
           if self.image is None:
               self.image = Image(self.filename)
           self.image.display()

4. Facade - Simplified interface to complex system
   class OrderFacade:
       def place_order(self, order):
           inventory.check()
           payment.process()
           shipping.schedule()
           notification.send()
```

### **Behavioral Patterns** (Object interaction)

```
1. Strategy - Select algorithm at runtime
   class PaymentContext:
       def __init__(self, strategy):
           self.strategy = strategy
       
       def pay(self, amount):
           self.strategy.pay(amount)

2. Observer - Notify dependents of state changes
   class Subject:
       def __init__(self):
           self.observers = []
       
       def notify(self):
           for observer in self.observers:
               observer.update()

3. Command - Encapsulate request as object
   class Command:
       def execute(self):
           pass
   
   class CopyCommand(Command):
       def execute(self):
           # Copy logic

4. Template Method - Define skeleton, subclasses fill steps
   class DataProcessor:
       def process(self):
           self.read_data()
           self.transform_data()  # Subclass implements
           self.save_data()
```

---

## 🧩 **OOP Concepts Review**

```
┌─────────────────────────────────────────────┐
│ 1. Encapsulation                            │
│    Hide internal state, expose interface    │
│                                             │
│    class BankAccount:                       │
│        def __init__(self):                  │
│            self.__balance = 0  # Private    │
│                                             │
│        def deposit(self, amount):           │
│            if amount > 0:                   │
│                self.__balance += amount     │
│                                             │
│        def get_balance(self):               │
│            return self.__balance            │
│                                             │
│    ✅ Can't directly modify balance         │
│    ✅ Validation in deposit method          │
└─────────────────────────────────────────────┘

┌─────────────────────────────────────────────┐
│ 2. Inheritance                              │
│    Reuse code from parent class             │
│                                             │
│    class Animal:                            │
│        def eat(self):                       │
│            pass                             │
│                                             │
│    class Dog(Animal):  # Inherits eat()    │
│        def bark(self):                      │
│            pass                             │
│                                             │
│    ✅ Code reuse                            │
│    ✅ "is-a" relationship                   │
└─────────────────────────────────────────────┘

┌─────────────────────────────────────────────┐
│ 3. Polymorphism                             │
│    Same interface, different implementations│
│                                             │
│    class Shape:                             │
│        def area(self):                      │
│            pass                             │
│                                             │
│    class Circle(Shape):                     │
│        def area(self):                      │
│            return 3.14 * r * r              │
│                                             │
│    class Square(Shape):                     │
│        def area(self):                      │
│            return side * side               │
│                                             │
│    shapes = [Circle(), Square()]            │
│    for shape in shapes:                     │
│        print(shape.area())  # Polymorphism! │
│                                             │
│    ✅ Treat different types uniformly       │
└─────────────────────────────────────────────┘

┌─────────────────────────────────────────────┐
│ 4. Abstraction                              │
│    Hide complexity, show only essentials    │
│                                             │
│    from abc import ABC, abstractmethod      │
│                                             │
│    class PaymentGateway(ABC):               │
│        @abstractmethod                      │
│        def process(self, amount):           │
│            pass                             │
│                                             │
│    class Stripe(PaymentGateway):            │
│        def process(self, amount):           │
│            # Stripe-specific implementation │
│            pass                             │
│                                             │
│    ✅ Focus on "what", not "how"            │
│    ✅ Enforce contracts                     │
└─────────────────────────────────────────────┘
```

---

## 🎯 **Composition vs Inheritance**

```
❌ Inheritance Overuse:

class Animal:
    def move(self): pass

class FlyingAnimal(Animal):
    def fly(self): pass

class SwimmingAnimal(Animal):
    def swim(self): pass

class Duck(FlyingAnimal, SwimmingAnimal):  # Multiple inheritance mess
    pass

Problem: What about flying fish? Swimming birds?

✅ Composition (Preferred):

class Flyable:
    def fly(self): pass

class Swimmable:
    def swim(self): pass

class Duck:
    def __init__(self):
        self.flying = Flyable()  # Compose behaviors
        self.swimming = Swimmable()
    
    def fly(self):
        self.flying.fly()
    
    def swim(self):
        self.swimming.swim()

Benefits:
✅ Flexible (add/remove behaviors at runtime)
✅ No diamond problem (multiple inheritance)
✅ "has-a" relationship (more natural)

Rule of Thumb:
• Use Inheritance for "is-a" relationships (Dog is an Animal)
• Use Composition for "has-a" relationships (Car has an Engine)
```

---

## 📋 **Interview Preparation Checklist**

When designing a class, ask:

1. **SRP**: Does this class have one clear responsibility?
2. **OCP**: Can I add features without modifying existing code?
3. **LSP**: Can subclasses substitute the parent safely?
4. **ISP**: Are interfaces focused (no fat interfaces)?
5. **DIP**: Am I depending on abstractions, not concrete classes?

**Example Interview Flow:**

Interviewer: "Design a notification system."

You:
1. **Identify responsibilities**: NotificationSender, NotificationFormatter, NotificationLogger (SRP)
2. **Define abstraction**: NotificationChannel interface (DIP)
3. **Implement channels**: EmailChannel, SMSChannel, PushChannel (OCP - add new without modifying)
4. **Use composition**: NotificationService has NotificationChannel (Composition over inheritance)
5. **Apply patterns**: Strategy (channel selection), Observer (notify subscribers)

Code:
```python
class NotificationChannel(ABC):
    @abstractmethod
    def send(self, message):
        pass

class EmailChannel(NotificationChannel):
    def send(self, message):
        # Send email

class SMSChannel(NotificationChannel):
    def send(self, message):
        # Send SMS

class NotificationService:
    def __init__(self, channel: NotificationChannel):
        self.channel = channel  # DIP
    
    def notify(self, user, message):
        formatted = self.format(message)  # SRP (separate formatting)
        self.channel.send(formatted)  # OCP (any channel)
        self.log(user, formatted)  # SRP (separate logging)
```

**Remember: Explain WHY, not just HOW!**

"I'm using the Strategy pattern here because notification channels can change at runtime (email during business hours, SMS for urgent). This makes the system flexible and follows OCP - we can add Slack or WhatsApp channels without modifying existing code."

---

## 🏆 **SOLID Principles Quick Reference**

| Principle | Summary | Key Benefit |
|-----------|---------|-------------|
| **S**RP | One class = One responsibility | Easy to change |
| **O**CP | Open extension, closed modification | Safe to extend |
| **L**SP | Subclass = Substitute parent | No surprises |
| **I**SP | Small, focused interfaces | No unused dependencies |
| **D**IP | Depend on abstractions | Flexible, testable |

**Mnemonic: "SOLID code is SOLID (strong, reliable, maintainable)"**

## 🏢 Top LLD Interview Questions (FAANG Companies)

### **Most Asked LLD Problems by Company:**

#### 1. **Design Parking Lot** (Amazon, Microsoft, Google)
```
🎯 Requirements:
• Multiple floors
• Different spot sizes (Compact, Large, Handicapped)
• Entry/Exit gates
• Parking fee calculation
• Spot availability tracking

🏗️ Object-Oriented Design:

┌─────────────────────────────────────────────────┐
│ CLASS DIAGRAM                                    │
│                                                   │
│ ParkingLot                                       │
│ ├── floors: List[Floor]                          │
│ ├── entry_gates: List[Gate]                     │
│ ├── exit_gates: List[Gate]                      │
│ └── methods:                                     │
│     ├── park_vehicle(vehicle)                    │
│     ├── unpark_vehicle(ticket)                   │
│     └── calculate_fee(ticket)                    │
│                                                   │
│ Floor                                            │
│ ├── floor_number: int                            │
│ ├── spots: List[ParkingSpot]                     │
│ └── methods:                                     │
│     └── find_available_spot(vehicle_type)        │
│                                                   │
│ ParkingSpot (Abstract)                           │
│ ├── spot_id: str                                 │
│ ├── is_available: bool                           │
│ ├── vehicle: Vehicle                             │
│ └── subclasses:                                  │
│     ├── CompactSpot                              │
│     ├── LargeSpot                                │
│     └── HandicappedSpot                          │
│                                                   │
│ Vehicle (Abstract)                               │
│ ├── license_plate: str                           │
│ ├── vehicle_type: VehicleType                    │
│ └── subclasses:                                  │
│     ├── Car                                      │
│     ├── Truck                                    │
│     ├── Motorcycle                               │
│     └── Van                                      │
│                                                   │
│ Ticket                                           │
│ ├── ticket_id: str                               │
│ ├── entry_time: datetime                         │
│ ├── exit_time: datetime                          │
│ ├── spot: ParkingSpot                            │
│ └── fee: float                                   │
│                                                   │
│ PaymentStrategy (Interface)                      │
│ ├── calculate_fee(ticket)                        │
│ └── implementations:                             │
│     ├── HourlyPayment                            │
│     ├── DailyPayment                             │
│     └── MonthlyPass                              │
└─────────────────────────────────────────────────┘

🎨 Design Patterns Used:
1. **Singleton**: ParkingLot instance
2. **Factory**: Create different vehicle types
3. **Strategy**: Different payment strategies
4. **Observer**: Notify when spots available

🔑 Key Interview Points:
• Enum for VehicleType (COMPACT, LARGE, MOTORCYCLE)
• SOLID principles (Single Responsibility)
• Inheritance vs Composition trade-off
• Thread safety for concurrent parking
• Database schema design

💡 Follow-up Questions:
Q: How to handle multiple floors efficiently?
A: HashMap<VehicleType, List<Floor>> for O(1) lookup

Q: What if parking lot is full?
A: Priority queue for waiting vehicles

Q: How to reserve spots in advance?
A: Add reservation_time field, cron job to expire
```

#### 2. **Design Elevator System** (Google, Microsoft, Uber)
```
🎯 Requirements:
• Multiple elevators
• Multiple floors (ground + n floors)
• Request from outside (up/down button)
• Request from inside (floor number)
• Optimize wait time

🏗️ Design:

┌─────────────────────────────────────────────────┐
│ Elevator                                         │
│ ├── id: int                                      │
│ ├── current_floor: int                           │
│ ├── direction: Direction (UP/DOWN/IDLE)          │
│ ├── requests: PriorityQueue[Request]             │
│ ├── capacity: int                                │
│ ├── current_load: int                            │
│ └── methods:                                     │
│     ├── move()                                   │
│     ├── open_door()                              │
│     ├── close_door()                             │
│     └── add_request(request)                     │
│                                                   │
│ ElevatorController                               │
│ ├── elevators: List[Elevator]                    │
│ ├── strategy: SchedulingStrategy                 │
│ └── methods:                                     │
│     └── assign_elevator(request)                 │
│                                                   │
│ Request                                          │
│ ├── source_floor: int                            │
│ ├── destination_floor: int                       │
│ ├── direction: Direction                         │
│ └── timestamp: datetime                          │
│                                                   │
│ SchedulingStrategy (Interface)                   │
│ └── implementations:                             │
│     ├── NearestCarStrategy                       │
│     │   → Assign nearest elevator                │
│     ├── DirectionBasedStrategy                   │
│     │   → Assign elevator going same direction   │
│     └── LoadBalancingStrategy                    │
│         → Distribute load evenly                 │
└─────────────────────────────────────────────────┘

🎯 Algorithm (SCAN/LOOK):
┌─────────────────────────────────────────────────┐
│ Elevator at Floor 5, going UP                    │
│ Requests: [7, 3, 10, 2, 8]                       │
│                                                   │
│ Step 1: Serve UP requests first                  │
│   Current: 5 → Next: 7 (pick up)                │
│   → 8 (pick up) → 10 (pick up)                   │
│                                                   │
│ Step 2: Change direction to DOWN                 │
│   Current: 10 → Next: 3 (pick up)                │
│   → 2 (pick up)                                  │
│                                                   │
│ This minimizes reversals (efficient!)            │
└─────────────────────────────────────────────────┘

🔑 Key Interview Points:
• State machine (MOVING_UP, MOVING_DOWN, IDLE)
• Priority queue for requests
• Starvation prevention (max wait time)
• Thread safety (synchronized methods)

💡 Follow-up Questions:
Q: How to handle emergency (fire alarm)?
A: State pattern - EmergencyState (go to ground floor)

Q: Optimize for peak hours (9am rush)?
A: Pre-position elevators (3 at ground, 2 at top floors)

Q: Power optimization?
A: Idle timeout - turn off lights, reduce movement
```

#### 3. **Design Chess Game** (Amazon, Google)
```
🎯 Requirements:
• Two players (White, Black)
• 8×8 board
• Different pieces (King, Queen, Rook, etc.)
• Move validation
• Check/Checkmate detection
• Castling, En passant special moves

🏗️ Object-Oriented Design:

┌─────────────────────────────────────────────────┐
│ ChessGame                                        │
│ ├── board: Board                                 │
│ ├── players: [Player, Player]                    │
│ ├── current_turn: Player                         │
│ ├── status: GameStatus (ACTIVE/CHECK/CHECKMATE) │
│ ├── move_history: List[Move]                     │
│ └── methods:                                     │
│     ├── make_move(from, to)                      │
│     ├── is_valid_move(move)                      │
│     ├── is_check()                               │
│     ├── is_checkmate()                           │
│     └── undo_move()                              │
│                                                   │
│ Board                                            │
│ ├── cells: Cell[8][8]                            │
│ └── methods:                                     │
│     ├── get_piece(row, col)                      │
│     └── set_piece(row, col, piece)               │
│                                                   │
│ Cell                                             │
│ ├── row: int (0-7)                               │
│ ├── col: int (0-7)                               │
│ ├── piece: Piece | None                          │
│                                                   │
│ Piece (Abstract)                                 │
│ ├── color: Color (WHITE/BLACK)                   │
│ ├── is_killed: bool                              │
│ ├── get_possible_moves(board) → List[Cell]      │
│ └── subclasses:                                  │
│     ├── King (1 square any direction)            │
│     ├── Queen (any direction, any distance)      │
│     ├── Rook (horizontal/vertical)               │
│     ├── Bishop (diagonal)                        │
│     ├── Knight (L-shape)                         │
│     └── Pawn (forward, capture diagonal)         │
│                                                   │
│ Move                                             │
│ ├── player: Player                               │
│ ├── piece: Piece                                 │
│ ├── from_cell: Cell                              │
│ ├── to_cell: Cell                                │
│ ├── piece_killed: Piece | None                   │
│ └── timestamp: datetime                          │
└─────────────────────────────────────────────────┘

🎨 Design Patterns:
1. **Command Pattern**: Move as command (undo/redo)
2. **Template Method**: Piece validation logic
3. **Factory**: Create pieces
4. **Singleton**: ChessGame instance

🔑 Move Validation (Example - Knight):
```python
class Knight(Piece):
    def get_possible_moves(self, board):
        moves = []
        row, col = self.current_cell.row, self.current_cell.col
        
        # L-shaped moves (8 possibilities)
        offsets = [
            (-2, -1), (-2, 1), (-1, -2), (-1, 2),
            (1, -2), (1, 2), (2, -1), (2, 1)
        ]
        
        for dr, dc in offsets:
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 8 and 0 <= new_col < 8:
                target_cell = board.cells[new_row][new_col]
                # Can move if empty or opponent's piece
                if not target_cell.piece or \
                   target_cell.piece.color != self.color:
                    moves.append(target_cell)
        
        return moves
```

💡 Follow-up Questions:
Q: How to detect checkmate?
A: 1) King in check, 2) No valid moves for any piece

Q: How to handle special moves (castling)?
A: Special validation in King class, check if rook hasn't moved

Q: Implement AI opponent?
A: Minimax algorithm with alpha-beta pruning (depth 4-6)
```

#### 4. **Design LRU Cache** (Facebook, Amazon, Google)
```
🎯 Requirements:
• get(key) - O(1)
• put(key, value) - O(1)
• Fixed capacity
• Evict least recently used when full

🏗️ Data Structure Design:

┌─────────────────────────────────────────────────┐
│ LRU Cache Architecture                           │
│                                                   │
│ HashMap + Doubly Linked List                     │
│                                                   │
│ HashMap: key → Node                              │
│ ┌─────┬──────┐                                   │
│ │ "a" │ Node1│ ─┐                                │
│ │ "b" │ Node2│ ─┼─┐                              │
│ │ "c" │ Node3│ ─┼─┼─┐                            │
│ └─────┴──────┘  │ │ │                            │
│                 │ │ │                            │
│ Doubly Linked List: (Most → Least recent)        │
│                 ▼ ▼ ▼                            │
│ Head ←→ [Node1] ←→ [Node2] ←→ [Node3] ←→ Tail   │
│ (dummy)   "a":1     "b":2     "c":3   (dummy)   │
│                                                   │
│ Operations:                                      │
│ • get("b"): Move Node2 to front                  │
│ • put("d", 4): Add at front, remove Tail.prev    │
└─────────────────────────────────────────────────┘

🎨 Implementation (Python):
```python
class Node:
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.prev = None
        self.next = None

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}  # key → Node
        self.head = Node(0, 0)  # Dummy head
        self.tail = Node(0, 0)  # Dummy tail
        self.head.next = self.tail
        self.tail.prev = self.head
    
    def _remove(self, node):
        """Remove node from linked list"""
        prev_node = node.prev
        next_node = node.next
        prev_node.next = next_node
        next_node.prev = prev_node
    
    def _add_to_front(self, node):
        """Add node right after head (most recent)"""
        node.next = self.head.next
        node.prev = self.head
        self.head.next.prev = node
        self.head.next = node
    
    def get(self, key: int) -> int:
        if key in self.cache:
            node = self.cache[key]
            self._remove(node)
            self._add_to_front(node)
            return node.value
        return -1
    
    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            # Update existing
            node = self.cache[key]
            node.value = value
            self._remove(node)
            self._add_to_front(node)
        else:
            # Add new
            if len(self.cache) >= self.capacity:
                # Remove least recently used (before tail)
                lru = self.tail.prev
                self._remove(lru)
                del self.cache[lru.key]
            
            new_node = Node(key, value)
            self.cache[key] = new_node
            self._add_to_front(new_node)

# Usage
cache = LRUCache(2)
cache.put(1, 1)  # cache: {1=1}
cache.put(2, 2)  # cache: {1=1, 2=2}
cache.get(1)     # returns 1, cache: {2=2, 1=1}
cache.put(3, 3)  # evicts 2, cache: {1=1, 3=3}
cache.get(2)     # returns -1 (not found)
```

💡 Follow-up Questions:
Q: Implement LFU (Least Frequently Used) cache?
A: Add frequency counter, min heap for O(log n)

Q: Thread-safe LRU cache?
A: Add ReentrantLock, synchronized methods

Q: Distributed LRU cache?
A: Redis (server-side), consistent hashing for sharding
```

#### 5. **Design Rate Limiter** (Stripe, Twitter, Amazon API Gateway)
```
🎯 Requirements:
• Limit API requests per user
• Different limits (100/min, 1000/hour)
• Distributed system support
• Minimal latency (<10ms)

🏗️ Algorithms:

┌─────────────────────────────────────────────────┐
│ 1. Token Bucket (Stripe's approach)             │
│                                                   │
│ Bucket: 100 tokens (capacity)                   │
│ Refill: 10 tokens/second                         │
│                                                   │
│ [🪙🪙🪙🪙🪙🪙🪙🪙🪙🪙] (10 tokens)                │
│ ↓ Request arrives                                │
│ [🪙🪙🪙🪙🪙🪙🪙🪙🪙] (9 tokens left)               │
│                                                   │
│ Every second: Add 10 tokens (max 100)            │
│                                                   │
│ Pros:                                            │
│ ✅ Smooth traffic                                │
│ ✅ Burst allowed (consume all tokens)            │
│ ✅ Simple to implement                           │
│                                                   │
│ Cons:                                            │
│ ❌ Token count in memory (state)                 │
└─────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────┐
│ 2. Sliding Window Log (Twitter's approach)       │
│                                                   │
│ Store timestamp of each request                  │
│ Window: Last 60 seconds                          │
│                                                   │
│ Request timestamps: [10:00:01, 10:00:05, ...]   │
│                                                   │
│ New request at 10:01:02:                         │
│ 1. Remove timestamps < 10:00:02                  │
│ 2. Count remaining (98 requests)                 │
│ 3. If count < 100: Allow                         │
│ 4. Else: Reject (429 Too Many Requests)          │
│                                                   │
│ Pros:                                            │
│ ✅ Accurate                                      │
│ ✅ No burst issues                               │
│                                                   │
│ Cons:                                            │
│ ❌ Memory intensive (store all timestamps)       │
└─────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────┐
│ 3. Fixed Window Counter (Simple)                 │
│                                                   │
│ Window: 10:00:00 - 10:01:00                      │
│ Counter: 98/100 requests                         │
│                                                   │
│ At 10:01:00: Reset counter to 0                  │
│                                                   │
│ Problem: Edge case burst                         │
│ 10:00:59 → 50 requests                           │
│ 10:01:01 → 50 requests                           │
│ = 100 requests in 2 seconds! (should be 60s)     │
│                                                   │
│ Pros:                                            │
│ ✅ Memory efficient                              │
│ ✅ Simple                                        │
│                                                   │
│ Cons:                                            │
│ ❌ Burst at window boundaries                    │
└─────────────────────────────────────────────────┘

🎨 Implementation (Token Bucket in Redis):
```python
import time
import redis

class TokenBucketRateLimiter:
    def __init__(self, redis_client, capacity=100, refill_rate=10):
        self.redis = redis_client
        self.capacity = capacity
        self.refill_rate = refill_rate  # tokens/second
    
    def allow_request(self, user_id):
        key = f"rate_limit:{user_id}"
        current_time = time.time()
        
        # Lua script for atomic operation
        lua_script = """
        local key = KEYS[1]
        local capacity = tonumber(ARGV[1])
        local refill_rate = tonumber(ARGV[2])
        local current_time = tonumber(ARGV[3])
        
        local token_data = redis.call('HMGET', key, 'tokens', 'last_refill')
        local tokens = tonumber(token_data[1]) or capacity
        local last_refill = tonumber(token_data[2]) or current_time
        
        -- Calculate new tokens
        local time_passed = current_time - last_refill
        local new_tokens = math.min(capacity, tokens + time_passed * refill_rate)
        
        if new_tokens >= 1 then
            -- Allow request
            new_tokens = new_tokens - 1
            redis.call('HSET', key, 'tokens', new_tokens, 'last_refill', current_time)
            redis.call('EXPIRE', key, 3600)  -- 1 hour TTL
            return 1  -- Allowed
        else
            return 0  -- Rate limited
        end
        """
        
        result = self.redis.eval(
            lua_script, 1, key,
            self.capacity, self.refill_rate, current_time
        )
        
        return bool(result)

# Usage
redis_client = redis.Redis(host='localhost', port=6379)
limiter = TokenBucketRateLimiter(redis_client)

if limiter.allow_request("user:123"):
    # Process request
    print("Request allowed")
else:
    # Return 429 Too Many Requests
    print("Rate limit exceeded")
```

💡 Distributed Rate Limiting:
• Use Redis (centralized counter)
• Or Cassandra (distributed counter)
• Consistent hashing for sharding

🔑 Interview Tips:
• Discuss trade-offs (accuracy vs memory)
• Mention Redis Lua scripts (atomic)
• Talk about distributed systems challenges
• Explain 429 status code
```

#### 6. **Design Notification Service** (Facebook, LinkedIn)
```
🎯 Requirements:
• Multiple channels (Email, SMS, Push)
• Priority levels (High, Medium, Low)
• Retry logic for failures
• Rate limiting (don't spam users)
• Template support

🏗️ Architecture:

┌─────────────────────────────────────────────────┐
│ NotificationService                              │
│ ├── send(notification)                           │
│ ├── retry_failed()                               │
│ └── get_user_preferences(user_id)                │
│                                                   │
│ Notification                                     │
│ ├── id: UUID                                     │
│ ├── user_id: str                                 │
│ ├── type: NotificationType (EMAIL/SMS/PUSH)      │
│ ├── priority: Priority (HIGH/MEDIUM/LOW)         │
│ ├── template: Template                           │
│ ├── content: dict                                │
│ └── retry_count: int                             │
│                                                   │
│ NotificationChannel (Interface)                  │
│ ├── send(notification)                           │
│ └── implementations:                             │
│     ├── EmailChannel (SendGrid/SES)              │
│     ├── SMSChannel (Twilio)                      │
│     └── PushChannel (FCM/APNs)                   │
│                                                   │
│ Template                                         │
│ ├── template_id: str                             │
│ ├── subject: str                                 │
│ ├── body: str (with placeholders)                │
│ └── render(data) → str                           │
│                                                   │
│ UserPreferences                                  │
│ ├── user_id: str                                 │
│ ├── enabled_channels: Set[NotificationType]      │
│ ├── quiet_hours: (start_time, end_time)          │
│ └── frequency_limit: int (max per day)           │
└─────────────────────────────────────────────────┘

🎨 Design Patterns:
1. **Strategy Pattern**: Different channels
2. **Template Method**: Notification sending flow
3. **Observer Pattern**: Subscribe to events
4. **Queue Pattern**: Async processing

🔄 Flow with Retry Logic:
```
1. Event occurs (new follower)
2. Create notification
3. Add to priority queue (Kafka/RabbitMQ)
4. Worker picks from queue
5. Check user preferences (quiet hours?)
6. Select channel (email, push, sms)
7. Attempt send
8. If success: Mark delivered
9. If failure: Retry with exponential backoff
   - Retry 1: After 1 minute
   - Retry 2: After 5 minutes  
   - Retry 3: After 30 minutes
   - Max retries: 3
10. After max retries: Move to dead letter queue
```

💡 Interview Points:
• Discuss deduplication (don't send same notif twice)
• Rate limiting per user (max 10 emails/day)
• Priority queue (high priority first)
• Idempotency (handle duplicate events)
```

---

## 🎓 Top 10 Design Patterns for Interviews

### **Creational Patterns:**
1. **Singleton** - One instance (Database connection pool)
2. **Factory** - Create objects (Vehicle factory)
3. **Builder** - Step-by-step construction (SQL query builder)

### **Structural Patterns:**
4. **Adapter** - Interface compatibility (Old API → New API)
5. **Decorator** - Add functionality (Logging decorator)
6. **Facade** - Simplified interface (Payment facade)

### **Behavioral Patterns:**
7. **Strategy** - Interchangeable algorithms (Sorting, Payment)
8. **Observer** - Event notification (Pub/Sub)
9. **Command** - Encapsulate requests (Undo/Redo)
10. **State** - State-based behavior (Order states)

---

## 📝 LLD Interview Template

```
Step 1: Clarify Requirements (5 min)
• Functional requirements (what features?)
• Non-functional (scalability, performance)
• Constraints (single-threaded, distributed?)

Step 2: Identify Objects/Classes (10 min)
• Nouns → Classes (User, Product, Order)
• Verbs → Methods (create, update, delete)
• Relationships (has-a, is-a)

Step 3: Define Class Diagram (15 min)
• Attributes and methods
• Inheritance hierarchy
• Interfaces
• Design patterns

Step 4: Implement Core Logic (15 min)
• Write pseudo-code or actual code
• Show SOLID principles
• Handle edge cases

Step 5: Discuss Trade-offs (5 min)
• Performance vs Code simplicity
• Extensibility vs Complexity
• Memory vs CPU
```

---

## 🔥 Practice Strategy

### **Week 1-2: SOLID + Patterns**
- Study all SOLID principles with examples
- Implement 10 design patterns

### **Week 3-4: Classic Problems**
- Parking Lot
- Elevator System
- Chess Game
- LRU Cache
- Rate Limiter

### **Week 5-6: Advanced Problems**
- Design Amazon (e-commerce)
- Design Uber (ride-sharing)
- Design Splitwise (expense sharing)
- Design BookMyShow (ticket booking)

### **Pro Tips:**
✅ Think out loud during interviews
✅ Ask clarifying questions
✅ Draw class diagrams
✅ Code in preferred language (Python/Java/C++)
✅ Test with examples
✅ Discuss extensibility

**Remember: LLD is about demonstrating clean code principles and design thinking, not perfect syntax!**

In [ ]:
# SOLID Principles Implementation - CAREER-DEFINING KNOWLEDGE!

print("🎯 SOLID PRINCIPLES - SOFTWARE ARCHITECTURE EXCELLENCE!")
print("=" * 60)

# ===============================================================
# 1. SINGLE RESPONSIBILITY PRINCIPLE (SRP)
# ===============================================================
print("\n🔥 1. SINGLE RESPONSIBILITY PRINCIPLE")
print("Each class should have only ONE reason to change")

# ❌ BAD EXAMPLE - Multiple responsibilities
class BadUser:
    def __init__(self, name, email):
        self.name = name
        self.email = email
    
    def save_to_database(self):
        """Database logic - responsibility 1"""
        print(f"Saving {self.name} to database")
    
    def send_email(self):
        """Email logic - responsibility 2"""
        print(f"Sending email to {self.email}")
    
    def validate_email(self):
        """Validation logic - responsibility 3"""
        return "@" in self.email

# ✅ GOOD EXAMPLE - Single responsibility per class
class User:
    """Responsible ONLY for user data"""
    def __init__(self, name, email):
        self.name = name
        self.email = email

class UserRepository:
    """Responsible ONLY for database operations"""
    def save(self, user):
        print(f"Saving {user.name} to database")
    
    def find_by_email(self, email):
        print(f"Finding user by email: {email}")

class EmailService:
    """Responsible ONLY for email operations"""
    def send_welcome_email(self, user):
        print(f"Sending welcome email to {user.email}")

class EmailValidator:
    """Responsible ONLY for email validation"""
    @staticmethod
    def is_valid(email):
        return "@" in email and "." in email

# Demonstration
user = User("John Doe", "john@example.com")
repository = UserRepository()
email_service = EmailService()
validator = EmailValidator()

print("\nSRP in action:")
print(f"User created: {user.name}")
print(f"Email valid: {validator.is_valid(user.email)}")
repository.save(user)
email_service.send_welcome_email(user)

# ===============================================================
# 2. OPEN/CLOSED PRINCIPLE (OCP)
# ===============================================================
print("\n\n🔥 2. OPEN/CLOSED PRINCIPLE")
print("Software entities should be OPEN for extension, CLOSED for modification")

from abc import ABC, abstractmethod

# ✅ GOOD EXAMPLE - Extensible without modification
class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height
    
    def area(self):
        return self.width * self.height

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    
    def area(self):
        return 3.14159 * self.radius ** 2

class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height
    
    def area(self):
        return 0.5 * self.base * self.height

class AreaCalculator:
    """Can handle any Shape without modification!"""
    def calculate_total_area(self, shapes):
        return sum(shape.area() for shape in shapes)

# Demonstration
shapes = [
    Rectangle(5, 10),
    Circle(3),
    Triangle(4, 6)
]

calculator = AreaCalculator()
total_area = calculator.calculate_total_area(shapes)
print(f"\nOCP in action:")
print(f"Total area of all shapes: {total_area:.2f}")

# ===============================================================
# 3. LISKOV SUBSTITUTION PRINCIPLE (LSP)
# ===============================================================
print("\n\n🔥 3. LISKOV SUBSTITUTION PRINCIPLE")
print("Objects of a superclass should be replaceable with objects of subclasses")

class Bird(ABC):
    @abstractmethod
    def move(self):
        pass

class FlyingBird(Bird):
    @abstractmethod
    def fly(self):
        pass

class Eagle(FlyingBird):
    def move(self):
        return "Flying high"
    
    def fly(self):
        return "Soaring through the sky"

class Penguin(Bird):
    def move(self):
        return "Swimming gracefully"

class BirdHandler:
    def make_bird_move(self, bird: Bird):
        return bird.move()

# Demonstration
birds = [Eagle(), Penguin()]
handler = BirdHandler()

print(f"\nLSP in action:")
for bird in birds:
    print(f"{bird.__class__.__name__}: {handler.make_bird_move(bird)}")

# ===============================================================
# 4. INTERFACE SEGREGATION PRINCIPLE (ISP)
# ===============================================================
print("\n\n🔥 4. INTERFACE SEGREGATION PRINCIPLE")
print("No client should be forced to depend on methods it does not use")

# ✅ GOOD EXAMPLE - Segregated interfaces
class Readable(ABC):
    @abstractmethod
    def read(self):
        pass

class Writable(ABC):
    @abstractmethod
    def write(self, data):
        pass

class Executable(ABC):
    @abstractmethod
    def execute(self):
        pass

class ReadOnlyFile(Readable):
    def __init__(self, filename):
        self.filename = filename
    
    def read(self):
        return f"Reading from {self.filename}"

class WriteableFile(Readable, Writable):
    def __init__(self, filename):
        self.filename = filename
    
    def read(self):
        return f"Reading from {self.filename}"
    
    def write(self, data):
        return f"Writing '{data}' to {self.filename}"

class ExecutableFile(Readable, Executable):
    def __init__(self, filename):
        self.filename = filename
    
    def read(self):
        return f"Reading from {self.filename}"
    
    def execute(self):
        return f"Executing {self.filename}"

# Demonstration
files = [
    ReadOnlyFile("document.txt"),
    WriteableFile("notes.txt"), 
    ExecutableFile("script.py")
]

print(f"\nISP in action:")
for file in files:
    print(f"{file.__class__.__name__}: {file.read()}")
    if isinstance(file, Writable):
        print(f"  Can write: {file.write('sample data')}")
    if isinstance(file, Executable):
        print(f"  Can execute: {file.execute()}")

# ===============================================================
# 5. DEPENDENCY INVERSION PRINCIPLE (DIP)
# ===============================================================
print("\n\n🔥 5. DEPENDENCY INVERSION PRINCIPLE")
print("High-level modules should not depend on low-level modules. Both should depend on abstractions.")

# ✅ GOOD EXAMPLE - Depend on abstractions
class NotificationService(ABC):
    @abstractmethod
    def send(self, message, recipient):
        pass

class EmailNotification(NotificationService):
    def send(self, message, recipient):
        return f"Email sent to {recipient}: {message}"

class SMSNotification(NotificationService):
    def send(self, message, recipient):
        return f"SMS sent to {recipient}: {message}"

class SlackNotification(NotificationService):
    def send(self, message, recipient):
        return f"Slack message sent to {recipient}: {message}"

class OrderService:
    """High-level module depends on abstraction, not concrete implementations"""
    def __init__(self, notification_service: NotificationService):
        self.notification_service = notification_service
    
    def process_order(self, order_id, customer):
        # Process order logic here...
        message = f"Order {order_id} has been processed successfully!"
        return self.notification_service.send(message, customer)

# Demonstration
print(f"\nDIP in action:")

# Can easily switch notification methods without changing OrderService
email_service = EmailNotification()
sms_service = SMSNotification()
slack_service = SlackNotification()

order_services = [
    OrderService(email_service),
    OrderService(sms_service),
    OrderService(slack_service)
]

for i, service in enumerate(order_services, 1):
    result = service.process_order(f"ORD-{i:03d}", "customer@example.com")
    print(f"  {result}")

# ===============================================================
# SOLID PRINCIPLES SUMMARY
# ===============================================================
print("\n" + "=" * 60)
print("🏆 SOLID PRINCIPLES MASTERY SUMMARY")
print("=" * 60)

principles = {
    "SRP": "One class, one responsibility - easier to maintain",
    "OCP": "Extend behavior without modifying existing code",
    "LSP": "Subclasses must be substitutable for their base classes",
    "ISP": "Many specific interfaces better than one general interface",
    "DIP": "Depend on abstractions, not concrete implementations"
}

for principle, description in principles.items():
    print(f"✅ {principle}: {description}")

print("\n💼 CAREER IMPACT:")
print("🏆 Essential for senior developer interviews")
print("🏆 Foundation for system architecture")
print("🏆 Demonstrates clean code expertise")
print("🏆 Required for technical leadership roles")
print("🏆 Gateway to software architect positions")

print("\n🚀 INTERVIEW TIP: Always explain WHY you're applying each principle!")
print("Show the problems they solve and the benefits they provide.")
print("=" * 60)

## Chapter 2: Gang of Four Design Patterns ⭐⭐⭐
> **System Design Interview Essential** - Master the 23 patterns that define software engineering excellence

### 🎯 Pattern Categories:
- **Creational**: Object creation mechanisms
- **Structural**: Object composition and relationships  
- **Behavioral**: Communication between objects

### 🚀 Career Impact:
- Required for all senior+ engineering interviews
- Foundation for system architecture decisions
- Gateway to principal engineer roles
- Essential for designing scalable systems

In [ ]:
# DESIGN PATTERNS - SOFTWARE ENGINEERING MASTERY!

print("🎯 GANG OF FOUR DESIGN PATTERNS - INTERVIEW GOLD!")
print("=" * 70)

from abc import ABC, abstractmethod
from typing import List, Dict, Any
from enum import Enum

# ===============================================================
# CREATIONAL PATTERNS - Object Creation Excellence
# ===============================================================

# 1. SINGLETON PATTERN - One instance to rule them all!
print("\n🔥 1. SINGLETON PATTERN")
print("Ensure a class has only one instance and provide global access")

class DatabaseConnection:
    _instance = None
    _is_initialized = False
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance
    
    def __init__(self):
        if not DatabaseConnection._is_initialized:
            self.connection_string = "mysql://localhost:3306/app"
            self.is_connected = False
            DatabaseConnection._is_initialized = True
    
    def connect(self):
        if not self.is_connected:
            print(f"Connecting to {self.connection_string}")
            self.is_connected = True
        return "Connected"
    
    def disconnect(self):
        if self.is_connected:
            print("Disconnecting from database")
            self.is_connected = False

# Test Singleton
db1 = DatabaseConnection()
db2 = DatabaseConnection()
print(f"Same instance? {db1 is db2}")  # True
print(f"Connection: {db1.connect()}")

# 2. FACTORY PATTERN - Create objects without specifying exact classes
print("\n\n🔥 2. FACTORY PATTERN")
print("Create objects without specifying their exact classes")

class PaymentProcessor(ABC):
    @abstractmethod
    def process_payment(self, amount: float) -> str:
        pass

class CreditCardProcessor(PaymentProcessor):
    def process_payment(self, amount: float) -> str:
        return f"Processing ${amount:.2f} via Credit Card"

class PayPalProcessor(PaymentProcessor):
    def process_payment(self, amount: float) -> str:
        return f"Processing ${amount:.2f} via PayPal"

class CryptoProcessor(PaymentProcessor):
    def process_payment(self, amount: float) -> str:
        return f"Processing ${amount:.2f} via Cryptocurrency"

class PaymentFactory:
    @staticmethod
    def create_processor(payment_type: str) -> PaymentProcessor:
        processors = {
            "credit_card": CreditCardProcessor,
            "paypal": PayPalProcessor,
            "crypto": CryptoProcessor
        }
        
        processor_class = processors.get(payment_type.lower())
        if not processor_class:
            raise ValueError(f"Unknown payment type: {payment_type}")
        
        return processor_class()

# Test Factory
payment_types = ["credit_card", "paypal", "crypto"]
for payment_type in payment_types:
    processor = PaymentFactory.create_processor(payment_type)
    result = processor.process_payment(100.0)
    print(f"  {result}")

# 3. BUILDER PATTERN - Construct complex objects step by step
print("\n\n🔥 3. BUILDER PATTERN")
print("Construct complex objects step by step")

class Computer:
    def __init__(self):
        self.cpu = None
        self.ram = None
        self.storage = None
        self.gpu = None
        self.os = None
    
    def __str__(self):
        specs = [
            f"CPU: {self.cpu}",
            f"RAM: {self.ram}",
            f"Storage: {self.storage}",
            f"GPU: {self.gpu}",
            f"OS: {self.os}"
        ]
        return "Computer Specs:\n" + "\n".join(f"  {spec}" for spec in specs)

class ComputerBuilder:
    def __init__(self):
        self.computer = Computer()
    
    def set_cpu(self, cpu: str):
        self.computer.cpu = cpu
        return self
    
    def set_ram(self, ram: str):
        self.computer.ram = ram
        return self
    
    def set_storage(self, storage: str):
        self.computer.storage = storage
        return self
    
    def set_gpu(self, gpu: str):
        self.computer.gpu = gpu
        return self
    
    def set_os(self, os: str):
        self.computer.os = os
        return self
    
    def build(self) -> Computer:
        return self.computer

# Test Builder
gaming_pc = (ComputerBuilder()
    .set_cpu("Intel i9-13900K")
    .set_ram("32GB DDR5")
    .set_storage("2TB NVMe SSD")
    .set_gpu("RTX 4080")
    .set_os("Windows 11")
    .build())

print(gaming_pc)

# ===============================================================
# STRUCTURAL PATTERNS - Object Composition Excellence
# ===============================================================

# 4. ADAPTER PATTERN - Make incompatible interfaces work together
print("\n\n🔥 4. ADAPTER PATTERN")
print("Allow incompatible interfaces to work together")

class OldPrinter:
    def old_print(self, text: str):
        return f"[OLD PRINTER] {text}"

class ModernPrinter:
    def print(self, text: str):
        return f"[MODERN PRINTER] {text}"

class PrinterAdapter:
    def __init__(self, old_printer: OldPrinter):
        self.old_printer = old_printer
    
    def print(self, text: str):
        return self.old_printer.old_print(text)

# Test Adapter
old_printer = OldPrinter()
adapted_printer = PrinterAdapter(old_printer)
modern_printer = ModernPrinter()

printers = [adapted_printer, modern_printer]
for i, printer in enumerate(printers, 1):
    result = printer.print(f"Document {i}")
    print(f"  {result}")

# 5. DECORATOR PATTERN - Add behavior to objects dynamically
print("\n\n🔥 5. DECORATOR PATTERN")
print("Add new functionality to objects without altering their structure")

class Coffee(ABC):
    @abstractmethod
    def cost(self) -> float:
        pass
    
    @abstractmethod
    def description(self) -> str:
        pass

class SimpleCoffee(Coffee):
    def cost(self) -> float:
        return 2.0
    
    def description(self) -> str:
        return "Simple coffee"

class CoffeeDecorator(Coffee):
    def __init__(self, coffee: Coffee):
        self._coffee = coffee

class MilkDecorator(CoffeeDecorator):
    def cost(self) -> float:
        return self._coffee.cost() + 0.5
    
    def description(self) -> str:
        return self._coffee.description() + ", milk"

class SugarDecorator(CoffeeDecorator):
    def cost(self) -> float:
        return self._coffee.cost() + 0.2
    
    def description(self) -> str:
        return self._coffee.description() + ", sugar"

class WhipDecorator(CoffeeDecorator):
    def cost(self) -> float:
        return self._coffee.cost() + 0.7
    
    def description(self) -> str:
        return self._coffee.description() + ", whipped cream"

# Test Decorator
coffee = SimpleCoffee()
coffee = MilkDecorator(coffee)
coffee = SugarDecorator(coffee)
coffee = WhipDecorator(coffee)

print(f"  Order: {coffee.description()}")
print(f"  Total cost: ${coffee.cost():.2f}")

# ===============================================================
# BEHAVIORAL PATTERNS - Object Communication Excellence
# ===============================================================

# 6. OBSERVER PATTERN - Define one-to-many dependency between objects
print("\n\n🔥 6. OBSERVER PATTERN")
print("Define a one-to-many dependency between objects")

class Subject:
    def __init__(self):
        self._observers: List['Observer'] = []
        self._state = None
    
    def attach(self, observer: 'Observer'):
        self._observers.append(observer)
    
    def detach(self, observer: 'Observer'):
        self._observers.remove(observer)
    
    def notify(self):
        for observer in self._observers:
            observer.update(self)
    
    def set_state(self, state):
        self._state = state
        self.notify()
    
    def get_state(self):
        return self._state

class Observer(ABC):
    @abstractmethod
    def update(self, subject: Subject):
        pass

class EmailNotifier(Observer):
    def update(self, subject: Subject):
        print(f"  📧 Email: Stock price changed to ${subject.get_state()}")

class SMSNotifier(Observer):
    def update(self, subject: Subject):
        print(f"  📱 SMS: Stock price updated to ${subject.get_state()}")

class AppNotifier(Observer):
    def update(self, subject: Subject):
        print(f"  📲 App: Push notification - Price is now ${subject.get_state()}")

# Test Observer
stock = Subject()
email_notifier = EmailNotifier()
sms_notifier = SMSNotifier()
app_notifier = AppNotifier()

stock.attach(email_notifier)
stock.attach(sms_notifier)
stock.attach(app_notifier)

print("  Stock price updates:")
stock.set_state(150.00)
stock.set_state(155.50)

# 7. STRATEGY PATTERN - Define family of algorithms
print("\n\n🔥 7. STRATEGY PATTERN")
print("Define a family of algorithms and make them interchangeable")

class SortStrategy(ABC):
    @abstractmethod
    def sort(self, data: List[int]) -> List[int]:
        pass

class BubbleSort(SortStrategy):
    def sort(self, data: List[int]) -> List[int]:
        arr = data.copy()
        n = len(arr)
        for i in range(n):
            for j in range(0, n - i - 1):
                if arr[j] > arr[j + 1]:
                    arr[j], arr[j + 1] = arr[j + 1], arr[j]
        return arr

class QuickSort(SortStrategy):
    def sort(self, data: List[int]) -> List[int]:
        arr = data.copy()
        self._quick_sort(arr, 0, len(arr) - 1)
        return arr
    
    def _quick_sort(self, arr, low, high):
        if low < high:
            pi = self._partition(arr, low, high)
            self._quick_sort(arr, low, pi - 1)
            self._quick_sort(arr, pi + 1, high)
    
    def _partition(self, arr, low, high):
        pivot = arr[high]
        i = low - 1
        for j in range(low, high):
            if arr[j] <= pivot:
                i += 1
                arr[i], arr[j] = arr[j], arr[i]
        arr[i + 1], arr[high] = arr[high], arr[i + 1]
        return i + 1

class SortContext:
    def __init__(self, strategy: SortStrategy):
        self._strategy = strategy
    
    def set_strategy(self, strategy: SortStrategy):
        self._strategy = strategy
    
    def sort(self, data: List[int]) -> List[int]:
        return self._strategy.sort(data)

# Test Strategy
data = [64, 34, 25, 12, 22, 11, 90]
print(f"  Original data: {data}")

context = SortContext(BubbleSort())
result1 = context.sort(data)
print(f"  Bubble sort: {result1}")

context.set_strategy(QuickSort())
result2 = context.sort(data)
print(f"  Quick sort: {result2}")

# 8. COMMAND PATTERN - Encapsulate requests as objects
print("\n\n🔥 8. COMMAND PATTERN")
print("Encapsulate a request as an object")

class Command(ABC):
    @abstractmethod
    def execute(self):
        pass
    
    @abstractmethod
    def undo(self):
        pass

class Light:
    def __init__(self, location: str):
        self.location = location
        self.is_on = False
    
    def turn_on(self):
        self.is_on = True
        return f"{self.location} light is ON"
    
    def turn_off(self):
        self.is_on = False
        return f"{self.location} light is OFF"

class LightOnCommand(Command):
    def __init__(self, light: Light):
        self.light = light
    
    def execute(self):
        return self.light.turn_on()
    
    def undo(self):
        return self.light.turn_off()

class LightOffCommand(Command):
    def __init__(self, light: Light):
        self.light = light
    
    def execute(self):
        return self.light.turn_off()
    
    def undo(self):
        return self.light.turn_on()

class RemoteControl:
    def __init__(self):
        self.commands: List[Command] = []
        self.last_command: Command = None
    
    def set_command(self, command: Command):
        self.commands.append(command)
    
    def press_button(self, slot: int):
        if 0 <= slot < len(self.commands):
            self.last_command = self.commands[slot]
            return self.last_command.execute()
        return "Invalid slot"
    
    def press_undo(self):
        if self.last_command:
            return self.last_command.undo()
        return "No command to undo"

# Test Command
living_room_light = Light("Living Room")
kitchen_light = Light("Kitchen")

light_on_cmd = LightOnCommand(living_room_light)
light_off_cmd = LightOffCommand(living_room_light)

remote = RemoteControl()
remote.set_command(light_on_cmd)
remote.set_command(light_off_cmd)

print(f"  {remote.press_button(0)}")  # Turn on
print(f"  {remote.press_button(1)}")  # Turn off
print(f"  {remote.press_undo()}")     # Undo (turn on)

# ===============================================================
# DESIGN PATTERNS SUMMARY
# ===============================================================
print("\n" + "=" * 70)
print("🏆 DESIGN PATTERNS MASTERY SUMMARY")
print("=" * 70)

pattern_benefits = {
    "Singleton": "Global access point, controlled instantiation",
    "Factory": "Loose coupling, easy to extend new types",
    "Builder": "Complex object construction, readable code",
    "Adapter": "Legacy system integration, interface compatibility",
    "Decorator": "Dynamic behavior addition, composition over inheritance",
    "Observer": "Loose coupling, event-driven architecture",
    "Strategy": "Algorithm interchangeability, runtime behavior change",
    "Command": "Request queuing, undo/redo functionality"
}

print("\n📚 PATTERNS IMPLEMENTED:")
for pattern, benefit in pattern_benefits.items():
    print(f"✅ {pattern}: {benefit}")

print("\n💼 CAREER IMPACT:")
print("🏆 Senior engineer interview essential")
print("🏆 System design foundation knowledge")
print("🏆 Code review and architecture discussions")
print("🏆 Team leadership and mentoring ability")
print("🏆 Path to software architect role")

print("\n🚀 INTERVIEW TIP: Always explain the PROBLEM each pattern solves!")
print("Show real-world scenarios where you'd use each pattern.")
print("Demonstrate understanding of trade-offs and alternatives.")
print("=" * 70)